### Import Dependencies

In [47]:
import openai
import pandas as pd

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery

### Create Qdrant collection for hybrid search

In [48]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [5]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-01-hybrid-search",
    vectors_config={
        "text-embedding-3-small": VectorParams(size=1536, distance=Distance.COSINE)
    },
    sparse_vectors_config={
        "bm25": SparseVectorParams(modifier=Modifier.IDF)
    }
)

True

In [49]:
qdrant_client.create_payload_index(
    collection_name="Amazon-items-collection-01-hybrid-search",
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=15, status=<UpdateStatus.COMPLETED: 'completed'>)

### Embedding Functions

In [50]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [51]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    
    return all_embeddings

### Read the sampled dataset with Amazon inventory data

In [52]:
df_items = pd.read_json(
    "../../data/meta_CDs_and_Vinyl_2022_2023_with_category_ratings_100_sample_1000.jsonl",
    lines=True
)

In [53]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Digital Music,"Girls, Girls, Girls",4.8,367,[],"[Girls, Girls, Girls — Mötley Crüe]",26.98,[{'thumb': 'https://m.media-amazon.com/images/...,[],Motley Crue Format: Vinyl,"[CDs & Vinyl, Rock, Hard Rock]",{'Product Dimensions': '0.16 x 12.32 x 12.48 i...,B09ZTFPVNB,NaN
1,Digital Music,"GarciaLive Vol. 18: November 2nd, 1974 - Keyst...",4.8,133,[],"[Produced from the original 1/4"" analog master...",12.49,[{'thumb': 'https://m.media-amazon.com/images/...,[],Jerry Garcia & Merl Saunders (Artist) Form...,"[CDs & Vinyl, Rock]",{'Product Dimensions': '5.59 x 0.39 x 4.92 inc...,B09YVCQC9V,NaN
2,Digital Music,RESIST,4.8,345,[],[],22.92,[{'thumb': 'https://m.media-amazon.com/images/...,[],Midnight Oil Format: Audio CD,"[CDs & Vinyl, AutoRip]",{'Product Dimensions': '4.9 x 5.4 x 0.3 inches...,B09MD1SRLF,NaN
3,Digital Music,Tana Talk 4,4.7,109,[],[2022 release. Benny the Butcher blazed a trai...,13.32,[{'thumb': 'https://m.media-amazon.com/images/...,[],Benny the Butcher Format: Audio CD,"[CDs & Vinyl, Rap & Hip-Hop]",{'Product Dimensions': '5.59 x 0.39 x 4.92 inc...,B09VPX9KFH,NaN
4,Digital Music,Chris Isaak Heart Shaped World (RSD Essential ...,4.7,409,[],[When Heart Shaped World was released in the s...,35.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Chris Isaak Format: Vinyl,"[CDs & Vinyl, Vinyl Store]",{'Product Dimensions': '12.2 x 0.2 x 12.2 inch...,B09TTSKVXY,NaN


### Preprocess title and features

In [54]:
def preprocess_description(row):
    return f"{row['title']} {' '.join(row['features'])}"

In [55]:
def extract_first_large_image(row):
    return row["images"][0].get("large", "")

In [56]:
df_items["preprocessed_description"] = df_items.apply(preprocess_description, axis=1)
df_items["image"] = df_items.apply(extract_first_large_image, axis=1)

In [57]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,preprocessed_description,image
0,Digital Music,"Girls, Girls, Girls",4.8,367,[],"[Girls, Girls, Girls — Mötley Crüe]",26.98,[{'thumb': 'https://m.media-amazon.com/images/...,[],Motley Crue Format: Vinyl,"[CDs & Vinyl, Rock, Hard Rock]",{'Product Dimensions': '0.16 x 12.32 x 12.48 i...,B09ZTFPVNB,NaN,"Girls, Girls, Girls",https://m.media-amazon.com/images/I/51yGOhceZU...
1,Digital Music,"GarciaLive Vol. 18: November 2nd, 1974 - Keyst...",4.8,133,[],"[Produced from the original 1/4"" analog master...",12.49,[{'thumb': 'https://m.media-amazon.com/images/...,[],Jerry Garcia & Merl Saunders (Artist) Form...,"[CDs & Vinyl, Rock]",{'Product Dimensions': '5.59 x 0.39 x 4.92 inc...,B09YVCQC9V,NaN,"GarciaLive Vol. 18: November 2nd, 1974 - Keyst...",https://m.media-amazon.com/images/I/51PU6YjqhO...
2,Digital Music,RESIST,4.8,345,[],[],22.92,[{'thumb': 'https://m.media-amazon.com/images/...,[],Midnight Oil Format: Audio CD,"[CDs & Vinyl, AutoRip]",{'Product Dimensions': '4.9 x 5.4 x 0.3 inches...,B09MD1SRLF,NaN,RESIST,https://m.media-amazon.com/images/I/31JpmDSjvM...
3,Digital Music,Tana Talk 4,4.7,109,[],[2022 release. Benny the Butcher blazed a trai...,13.32,[{'thumb': 'https://m.media-amazon.com/images/...,[],Benny the Butcher Format: Audio CD,"[CDs & Vinyl, Rap & Hip-Hop]",{'Product Dimensions': '5.59 x 0.39 x 4.92 inc...,B09VPX9KFH,NaN,Tana Talk 4,https://m.media-amazon.com/images/I/513cLiDTDS...
4,Digital Music,Chris Isaak Heart Shaped World (RSD Essential ...,4.7,409,[],[When Heart Shaped World was released in the s...,35.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Chris Isaak Format: Vinyl,"[CDs & Vinyl, Vinyl Store]",{'Product Dimensions': '12.2 x 0.2 x 12.2 inch...,B09TTSKVXY,NaN,Chris Isaak Heart Shaped World (RSD Essential ...,https://m.media-amazon.com/images/I/41awyiox6o...


In [58]:
list(df_items["preprocessed_description"].items())[0]

(0, 'Girls, Girls, Girls ')

In [59]:
list(df_items["image"].items())[0]

(0, 'https://m.media-amazon.com/images/I/51yGOhceZUL.jpg')

In [60]:
df_data_to_embed = df_items[["preprocessed_description", "image", "rating_number", "price", "average_rating", "parent_asin"]]

In [61]:
df_data_to_embed.head()

,preprocessed_description,image,rating_number,price,average_rating,parent_asin
0,"Girls, Girls, Girls",https://m.media-amazon.com/images/I/51yGOhceZU...,367,26.98,4.8,B09ZTFPVNB
1,"GarciaLive Vol. 18: November 2nd, 1974 - Keyst...",https://m.media-amazon.com/images/I/51PU6YjqhO...,133,12.49,4.8,B09YVCQC9V
2,RESIST,https://m.media-amazon.com/images/I/31JpmDSjvM...,345,22.92,4.8,B09MD1SRLF
3,Tana Talk 4,https://m.media-amazon.com/images/I/513cLiDTDS...,109,13.32,4.7,B09VPX9KFH
4,Chris Isaak Heart Shaped World (RSD Essential ...,https://m.media-amazon.com/images/I/41awyiox6o...,409,35.99,4.7,B09TTSKVXY


In [62]:
data_to_embed = df_data_to_embed.to_dict(orient="records")

In [63]:
data_to_embed

[{'preprocessed_description': 'Girls, Girls, Girls ',
  'image': 'https://m.media-amazon.com/images/I/51yGOhceZUL.jpg',
  'rating_number': 367,
  'price': 26.98,
  'average_rating': 4.8,
  'parent_asin': 'B09ZTFPVNB'},
 {'preprocessed_description': 'GarciaLive Vol. 18: November 2nd, 1974 - Keystone Berkeley[2 CD] ',
  'image': 'https://m.media-amazon.com/images/I/51PU6YjqhOL.jpg',
  'rating_number': 133,
  'price': 12.49,
  'average_rating': 4.8,
  'parent_asin': 'B09YVCQC9V'},
 {'preprocessed_description': 'RESIST ',
  'image': 'https://m.media-amazon.com/images/I/31JpmDSjvML.jpg',
  'rating_number': 345,
  'price': 22.92,
  'average_rating': 4.8,
  'parent_asin': 'B09MD1SRLF'},
 {'preprocessed_description': 'Tana Talk 4 ',
  'image': 'https://m.media-amazon.com/images/I/513cLiDTDSL.jpg',
  'rating_number': 109,
  'price': 13.32,
  'average_rating': 4.7,
  'parent_asin': 'B09VPX9KFH'},
 {'preprocessed_description': 'Chris Isaak Heart Shaped World (RSD Essential Edit ',
  'image': 'htt

In [27]:
len(data_to_embed)

2000

In [64]:
text_to_embed = [item["preprocessed_description"] for item in data_to_embed]

In [65]:
text_to_embed

['Girls, Girls, Girls ',
 'GarciaLive Vol. 18: November 2nd, 1974 - Keystone Berkeley[2 CD] ',
 'RESIST ',
 'Tana Talk 4 ',
 'Chris Isaak Heart Shaped World (RSD Essential Edit ',
 "Maybe You'Ve Been Brainwashed Too Black ",
 'No More Worlds To Conquer ',
 'Whitsitt Chapel ',
 'Time, Tequila & Therapy ',
 'Snow Waltz ',
 'Christmas ',
 'Inner Spirit: The 1979 Concert At The Teatro General San Martín[2 CD] ',
 'Cotton Mouth Man ',
 'A Neil Diamond Christmas ',
 'Mercury – Act 2[2 LP] ',
 'Hard To Find Jukebox: Stereo Explosion 7 ',
 'Listen To The Music ',
 'Carousel Of Time ',
 'Remember That You Will Die ',
 'All My Friends: Celebrating The Songs & Voice Of Gregg Allman [4 LP] ',
 "Tell Me That It's Over ",
 'Love Is The New Black Gold ',
 'Edge Of Thorns ',
 'A Family Christmas ',
 "Everybody Knows It's Christmas[Candy Floss LP] ",
 'In Cauda Venenum Extended Edition ',
 'LEGENDADDY[2 LP]       Explicit Lyrics ',
 'The Click Deluxe Pressing ',
 'Glitch Mode Photobook Version Random C

In [66]:
embeddings = get_embeddings_batch(text_to_embed)

Processed 100 of 2000
Processed 200 of 2000
Processed 300 of 2000
Processed 400 of 2000
Processed 500 of 2000
Processed 600 of 2000
Processed 700 of 2000
Processed 800 of 2000
Processed 900 of 2000
Processed 1000 of 2000
Processed 1100 of 2000
Processed 1200 of 2000
Processed 1300 of 2000
Processed 1400 of 2000
Processed 1500 of 2000
Processed 1600 of 2000
Processed 1700 of 2000
Processed 1800 of 2000
Processed 1900 of 2000
Processed 2000 of 2000


In [67]:
len(embeddings)

2000

In [73]:
pointstructs = []
i=1
for embedding, data in zip(embeddings, data_to_embed):
    pointstructs.append(
        PointStruct(
            id=i,
            vector={
                "text-embedding-3-small": embedding,
                "bm25": Document(
                    text=data["preprocessed_description"],
                    model="qdrant/bm25"
                )
            },
            payload=data
        )
    )
    i += 1

In [74]:
pointstructs[0].vector

{'text-embedding-3-small': [0.042724609375,
  0.05621337890625,
  -0.010101318359375,
  -0.00572967529296875,
  0.016204833984375,
  -0.024383544921875,
  0.026519775390625,
  -0.0190887451171875,
  -0.0170745849609375,
  -0.0032024383544921875,
  0.007793426513671875,
  0.002712249755859375,
  0.02935791015625,
  0.00855255126953125,
  -0.00327301025390625,
  0.015869140625,
  -0.010223388671875,
  -0.01104736328125,
  0.0157928466796875,
  0.037628173828125,
  0.0399169921875,
  0.052703857421875,
  0.00458526611328125,
  -0.046600341796875,
  0.05609130859375,
  0.0595703125,
  -0.00449371337890625,
  -0.0018320083618164062,
  -0.003498077392578125,
  0.0217132568359375,
  0.02777099609375,
  -0.0206298828125,
  -0.0015869140625,
  -0.014739990234375,
  -0.0341796875,
  -0.00023853778839111328,
  -0.044891357421875,
  0.022247314453125,
  0.0283355712890625,
  -0.0284271240234375,
  0.05560302734375,
  -0.0262451171875,
  0.01483917236328125,
  -0.0543212890625,
  0.0116424560546875

In [75]:
COLLECTION_NAME = "Amazon-items-collection-01-hybrid-search"
BATCH_SIZE = 500

for start in range(0, len(pointstructs), BATCH_SIZE):
    batch = pointstructs[start : start + BATCH_SIZE]
    qdrant_client.upsert(
        collection_name=COLLECTION_NAME,
        points=batch,
        wait=True,
    )
    print(f"Upserted {start + len(batch)} / {len(pointstructs)}")

Upserted 500 / 2000
Upserted 1000 / 2000
Upserted 1500 / 2000
Upserted 2000 / 2000


### Hybrid Retrieval

In [ ]:
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
/    }

In [77]:
results = retrieve_data("Best song of the summer?", k=20)

In [78]:
results

{'retrieved_context_ids': ['B09XGWCB4B',
  'B09XGWCB4B',
  'B09Y4X2XTS',
  'B09Y4X2XTS',
  'B09NFQ6593',
  'B09NFQ6593',
  'B0C1SWKV9J',
  'B0C1SWKV9J',
  'B0B14D2HZ6',
  'B09RG97953',
  'B09RG97953',
  'B0BLZG2YMG',
  'B0B14D2HZ6',
  'B0BLZG2YMG',
  'B09RRZYVPL',
  'B09RRZYVPL',
  'B0B28N7NKP',
  'B0B28N7NKP',
  'B0BTB6DFSW',
  'B0BTB6DFSW'],
 'retrieved_context': ['Sounds Of Summer: The Very Best Of The Beach Boys[Remastered] ',
  'Sounds Of Summer: The Very Best Of The Beach Boys[Remastered] ',
  'Songs About You ',
  'Songs About You ',
  'Summer Of Soul ...Or, When The Revolution Could Not Be Televised Soundtrack ',
  'Summer Of Soul ...Or, When The Revolution Could Not Be Televised Soundtrack ',
  'Stick Season       Explicit Lyrics ',
  'Stick Season       Explicit Lyrics ',
  'Donna Summer: 40th Anniversary Picture ',
  'Dawn FM by The Weekend Autograph       explicit_lyrics ',
  'Dawn FM by The Weekend Autograph       explicit_lyrics ',
  'Listen To The Music ',
  'Donna Summe

### Hybrid Search with weighted RRF

In [ ]:
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=models.RrfQuery(rrf=models.Rrf(weights=[3,1])),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [92]:
results = retrieve_data("whats a good charlie brown summer song?", k=20)

In [93]:
results

{'retrieved_context_ids': ['B09XSX514Q',
  'B0B9BX5C4K',
  'B0B9BX5C4K',
  'B09XSX514Q',
  'B0BFRRZPP8',
  'B0BFRRZPP8',
  'B09XGWCB4B',
  'B09XGWCB4B',
  'B09NFQ6593',
  'B09NFQ6593',
  'B09PK47CGG',
  'B09PK47CGG',
  'B0B832LSNC',
  'B0B832LSNC',
  'B0BBRNMGSZ',
  'B0BBRNMGSZ',
  'B0BFLT4TJT',
  'B0BFLT4TJT',
  'B09TQJ69KH',
  'B09TQJ69KH'],
 'retrieved_context': ["It's The Great Pumpkin, Charlie Brown[45rpm LP] ",
  'A Charlie Brown Christmas (Deluxe Edition) ',
  'A Charlie Brown Christmas (Deluxe Edition) ',
  "It's The Great Pumpkin, Charlie Brown[45rpm LP] ",
  'CHARLIE ',
  'CHARLIE ',
  'Sounds Of Summer: The Very Best Of The Beach Boys[Remastered] ',
  'Sounds Of Summer: The Very Best Of The Beach Boys[Remastered] ',
  'Summer Of Soul ...Or, When The Revolution Could Not Be Televised Soundtrack ',
  'Summer Of Soul ...Or, When The Revolution Could Not Be Televised Soundtrack ',
  'SLEEPLESS IN SEATTLE--Original Motion Picture Soundtrack (Sunset Vinyl Edition) ',
  'SLEEPLESS 